In [ ]:
import matplotlib.pyplot as plt
import numpy as np

decades = [1960, 1970, 1980, 1990, 2000, 2010]
region_colors = {
    'West Coast': '#2196F3',
    'Northwest / Mountain': '#4CAF50',
    'South': '#FF9800',
    'East Coast': '#9C27B0'
}

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(decades))
width = 0.2

for i, (region, color) in enumerate(region_colors.items()):
    means = []
    for dec in decades:
        vals = [v for yr, v in all_results['historical'][region].items()
                if dec <= yr < dec + 10]
        means.append(np.mean(vals) if vals else 0)
    ax.bar(x + i * width, means, width, label=region, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([str(d) + 's' for d in decades])
ax.set_ylabel('Average TCSI')
ax.set_title('How Has Tourism Climate Comfort Changed Each Decade?\nAverage TCSI by Region (Historical)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
ax.axhline(0.65, color='green', ls='--', alpha=0.4, label='Comfortable threshold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('graph1_decade_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

scenario_styles = {
    'historical': dict(color='black',   lw=2.5, ls='-',  label='Historical'),
    'ssp126':     dict(color='#2196F3', lw=2,   ls='-',  label='SSP1-2.6 (Low)'),
    'ssp245':     dict(color='#FF9800', lw=2,   ls='--', label='SSP2-4.5 (Mid)'),
    'ssp585':     dict(color='#F44336', lw=2,   ls=':',  label='SSP5-8.5 (High)'),
}

# Show just the US average (mean across all 4 regions)
for scen, style in scenario_styles.items():
    if scen not in all_results:
        continue
    all_years = sorted(set(yr for r in all_results[scen].values() for yr in r.index))
    avg = [np.mean([all_results[scen][r][yr]
                    for r in all_results[scen] if yr in all_results[scen][r]])
           for yr in all_years]
    smoothed = pd.Series(avg, index=all_years).rolling(5, center=True).mean()
    ax.plot(smoothed.index, smoothed.values, **style)

ax.axvline(2015, color='gray', lw=1, ls='--', alpha=0.5)
ax.text(2016, 0.3, 'Projections start →', color='gray', fontsize=9)
ax.fill_between([2015, 2100], 0, 1, alpha=0.04, color='red')
ax.set_title('When Do Emissions Scenarios Diverge?\nUS-Average TCSI Under Different Climate Futures', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Average TCSI (all regions)')
ax.set_ylim(0.2, 0.9)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('graph2_scenario_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Season offsets per region (same values used in TCSI calculation)
SEASON_MOD = {
    'annual': {'West Coast': 0,     'Northwest / Mountain': 0,    'South': 0,     'East Coast': 0    },
    'winter': {'West Coast':-0.04,  'Northwest / Mountain': 0.09, 'South': 0.13,  'East Coast':-0.13 },
    'spring': {'West Coast': 0.05,  'Northwest / Mountain': 0.03, 'South': 0.05,  'East Coast': 0.07 },
    'summer': {'West Coast': 0.07,  'Northwest / Mountain': 0.06, 'South':-0.14,  'East Coast': 0.04 },
    'fall':   {'West Coast': 0.03,  'Northwest / Mountain':-0.02, 'South': 0.06,  'East Coast': 0.02 },
}
seasons  = ['winter', 'spring', 'summer', 'fall']
regions  = list(region_colors.keys())

# Build matrix: rows = regions, cols = seasons, value = 2080 projected TCSI (ssp585)
matrix = np.zeros((len(regions), len(seasons)))
for ri, region in enumerate(regions):
    base_2080 = all_results['ssp585'][region].get(2080,
                all_results['ssp585'][region].iloc[-5:].mean())
    for si, season in enumerate(seasons):
        matrix[ri, si] = np.clip(base_2080 + SEASON_MOD[season][region], 0.05, 0.97)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0.2, vmax=0.85, aspect='auto')
ax.set_xticks(range(len(seasons)))
ax.set_xticklabels([s.capitalize() for s in seasons], fontsize=12)
ax.set_yticks(range(len(regions)))
ax.set_yticklabels(regions, fontsize=11)
plt.colorbar(im, ax=ax, label='TCSI (0=poor, 1=ideal)')

for ri in range(len(regions)):
    for si in range(len(seasons)):
        ax.text(si, ri, f'{matrix[ri,si]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='black' if 0.4 < matrix[ri,si] < 0.75 else 'white')

ax.set_title('Which Season Suffers Most by 2080?\nRegional TCSI by Season (SSP5-8.5 High Emissions)', fontweight='bold')
plt.tight_layout()
plt.savefig('graph3_seasonal_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
regions  = list(region_colors.keys())
scenarios = {'ssp126': '#2196F3', 'ssp245': '#FF9800', 'ssp585': '#F44336'}
years_compare = [2024, 2050, 2080]

fig, axes = plt.subplots(1, 3, figsize=(14, 6), sharey=True)

for ax, year in zip(axes, years_compare):
    scen_data = {scen: [] for scen in scenarios}
    for scen in scenarios:
        for region in regions:
            v = all_results[scen][region].get(year, np.nan)
            scen_data[scen].append(float(v) if not isinstance(v, float) else v)

    x = np.arange(len(regions))
    w = 0.25
    for i, (scen, color) in enumerate(scenarios.items()):
        ax.bar(x + i*w, scen_data[scen], w, label=scen.upper(), color=color, alpha=0.82)

    ax.set_title(f'Year {year}', fontweight='bold', fontsize=13)
    ax.set_xticks(x + w)
    ax.set_xticklabels(['W. Coast', 'NW/Mtn', 'South', 'E. Coast'], fontsize=9)
    ax.set_ylim(0, 1)
    ax.axhline(0.65, color='green', ls='--', alpha=0.35)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('TCSI')

axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
fig.suptitle('How Much Does Climate Change Shrink the Tourism Comfort Window?\nTCSI by Region, Scenario, and Future Year',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('graph4_region_year_comparison.png', dpi=150, bbox_inches='tight')
plt.show()